In [1]:
import pandas as pd

season = 2012

df = pd.read_parquet(fr'..\data\unprocessed\womens_sports_reference\full_season_sports_reference_{season}.parquet')

df = df.loc[df['NCAA Tournament'] == 0, :].reset_index(drop=True)  # only include games before NCAA Tournament

df

,Team,Date,NCAA Tournament,Location,Opponent,Type,Result,Team Score,Opponent Score,Team FG,...,Opponent TOV,Opponent PF,Overtimes Amount,Overtime,Score Differential,Adjusted Score Differential,Possessions,Team PPP,Opponent PPP,Tempo
0,Air Force,2011-11-11,0,-1,Lipscomb,REG (Non-Conf),1,68.0,59.0,23.0,...,23.0,24.0,0,0,9.0,1.361013,77.42,0.878326,0.762077,77.420000
1,Air Force,2011-11-13,0,-1,Tennessee-Martin,REG (Non-Conf),-1,60.0,84.0,27.0,...,18.0,7.0,0,0,-24.0,1.500000,81.80,0.733496,1.026895,81.800000
2,Air Force,2011-11-17,0,1,Denver,REG (Non-Conf),-1,52.0,54.0,20.0,...,20.0,19.0,0,0,-2.0,1.102120,78.00,0.666667,0.692308,78.000000
3,Air Force,2011-11-19,0,-1,Texas Southern,REG (Non-Conf),1,64.0,54.0,20.0,...,30.0,27.0,0,0,10.0,1.381279,80.54,0.794636,0.670474,80.540000
4,Air Force,2011-11-21,0,-1,Texas State,REG (Non-Conf),1,77.0,69.0,25.0,...,14.0,20.0,0,0,8.0,1.338710,77.22,0.997151,0.893551,77.220000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10251,Youngstown State,2012-02-18,0,1,Green Bay,REG (Conf),-1,72.0,77.0,21.0,...,14.0,19.0,1,1,-5.0,1.253292,73.40,0.980926,1.049046,65.244444
10252,Youngstown State,2012-02-23,0,-1,Wright State,REG (Conf),-1,62.0,84.0,21.0,...,10.0,15.0,0,0,-22.0,1.500000,73.98,0.838064,1.135442,73.980000
10253,Youngstown State,2012-02-25,0,-1,Detroit Mercy,REG (Conf),-1,64.0,67.0,20.0,...,10.0,14.0,0,0,-3.0,1.166625,62.76,1.019758,1.067559,62.760000
10254,Youngstown State,2012-03-03,0,-1,Cleveland State,REG (Conf),-1,69.0,79.0,24.0,...,24.0,22.0,0,0,-10.0,1.381279,86.84,0.794565,0.909719,86.840000


Filter to just wins because I don't need duplicate perspectives

In [2]:
df = df.loc[
    df['Result'] == 1, 
    ['Date', 'Team', 'Location', 'Opponent', 'Result', 'Adjusted Score Differential']
].reset_index(drop=True)

df.sort_values(['Date'], inplace=True, ignore_index=True)

df

,Date,Team,Location,Opponent,Result,Adjusted Score Differential
0,2011-11-11,Air Force,-1,Lipscomb,1,1.361013
1,2011-11-11,Iowa State,1,Houston Christian,1,1.500000
2,2011-11-11,James Madison,-1,Quinnipiac,1,1.338710
3,2011-11-11,Army,1,Wagner,1,1.500000
4,2011-11-11,American,-1,George Mason,1,1.214668
...,...,...,...,...,...,...
5123,2012-03-16,Minnesota,1,Charleston Southern,1,1.500000
5124,2012-03-16,Oklahoma State,1,Central Arkansas,1,1.500000
5125,2012-03-16,Toledo,1,Detroit Mercy,1,1.381279
5126,2012-03-16,Pacific,1,Arizona State,1,1.462122


In [3]:
def rescale_weights(arr, minimum, maximum):
    """
    Rescale the weights array to match desired weights.
    Assumes the data is already transformed where a 1 point win is 1.00, a blowout is 1.50, and a tie is 0.50.
    Ties will be weighted as half the minimum.
    """

    arr_scaled = ((arr - 1.00) / 0.50) * (maximum - minimum) + minimum
    arr_scaled[arr == 0.50] = minimum / 2
    return arr_scaled

In [4]:
from typing import Tuple
from openskill.models import BradleyTerryFull
import numpy as np

def get_season_ratings(X_train_weights: np.array, sigma: float, hfa_mu: float) -> Tuple[BradleyTerryFull, dict]:
    """
    Run the algorithm over the course of a season.

    Args:
        X_train_weights (np.array): Numpy array where each row is a game and columns are Winner, Loser, Location, Weight.
        sigma (float): The starting variance for each team rating.
        hfa_mu (float): The mu of home field advantage.

    Returns:
        Tuple[BradleyTerryFull, dict]: The environment object and dictionary of team ratings.
    """
    # initialize
    env = BradleyTerryFull(sigma=sigma)

    teams = set(X_train_weights[:, 0]).union(set(X_train_weights[:, 1]))
    team_ratings = dict(zip(teams, [env.rating() for _ in range(len(teams))]))
    hfa = env.rating(mu=hfa_mu, sigma=0.0)

    # iterate through games
    for winner, loser, location, weight in X_train_weights:
        winner_rating = team_ratings[winner]
        loser_rating = team_ratings[loser]
        if location == 1:
            [[winner_post, _], [loser_post]] = env.rate([[winner_rating, hfa], [loser_rating]])
        elif location == -1:
            [[winner_post], [loser_post, _]] = env.rate([[winner_rating], [loser_rating, hfa]])
        else:
            [[winner_post], [loser_post]] = env.rate([[winner_rating], [loser_rating]])

        winner_mu_adjustment = (winner_post.mu - winner_rating.mu)*weight
        winner_sigma_adjustment = (winner_post.sigma - winner_rating.sigma)*weight
        team_ratings[winner] = env.rating(mu=winner_rating.mu + winner_mu_adjustment, sigma=winner_rating.sigma + winner_sigma_adjustment)

        loser_mu_adjustment = (loser_post.mu - loser_rating.mu)*weight
        loser_sigma_adjustment = (loser_post.sigma - loser_rating.sigma)*weight
        team_ratings[loser] = env.rating(mu=loser_rating.mu + loser_mu_adjustment, sigma=loser_rating.sigma + loser_sigma_adjustment)

    return env, team_ratings

In [5]:
hfa_mu = 0.50
sigma = 25/3
minimum = 2/3
maximum = 4/3


X_train = df[['Team', 'Opponent', 'Location']].to_numpy()
weights = rescale_weights(df['Adjusted Score Differential'].to_numpy(), minimum, maximum)

X_train_weights = np.column_stack((X_train, weights))

env, team_ratings = get_season_ratings(X_train_weights, sigma, hfa_mu)

len(team_ratings)

342

In [6]:
df_ratings = pd.DataFrame(
    {
        'Team': team_ratings.keys(),
        'Mu': [v.mu for v in team_ratings.values()],
        'Sigma': [v.sigma for v in team_ratings.values()],
    }
)

df_ratings['OS Rating'] = df_ratings['Mu'] - df_ratings['Sigma']*3

df_ratings.sort_values(['OS Rating', 'Mu'], ascending=False, ignore_index=True, inplace=True)

df_ratings.head(50)

,Team,Mu,Sigma,OS Rating
0,Baylor,59.530745,4.380355,46.389680
1,Stanford,54.907487,4.414037,41.665374
2,Connecticut,53.834791,4.080311,41.593858
3,Notre Dame,54.117201,4.254520,41.353639
4,Delaware,52.593131,4.793309,38.213204
5,Tennessee,48.857011,3.936556,37.047344
6,Maryland,49.752533,4.334774,36.748210
7,Duke,47.741366,4.450309,34.390438
8,St. Bonaventure,46.650074,4.574590,32.926304
9,St. John's (NY),44.444476,4.071305,32.230561


In [7]:
df_ratings.to_parquet(f'../data/preprocessed/womens_os_rankings/os_rankings_{season}.parquet')

'Done'

'Done'